# GTFS Inspection Notebook

This notebook primarily inspects LA Metro's GTFS data, downloaded from two sources. We downloaded the recent (Week of October 19th, 2026) GTFS schedule data from their [official GTFS Schedule Data website](https://developer.metro.net/gtfs-schedule-data/). Meanwhile, we downloaded their 2020 GTFS data for the same time period from [MobilityDatabase](https://mobilitydatabase.org/feeds/gtfs/mdb-30).

## Data Dictionaries 
For someone who is using the GTFS files for the first time, you can refer to the GTFS data dictionaries named `GTFS_dict.md` in the main repository.

In short, each zip should have 9 to 11 `.txt` files, depending on the year of the data:

1. `agency.txt` - Transit agencies with service represented in this dataset.
2. `calendar_dates.txt` - Exceptions for the services defined in the `calendar.txt`.
3. `calendar.txt` - Service dates specified using a weekly schedule with start and end dates.
4. `feed_info.txt` - Dataset metadata, including publisher, version, and expiration information.
5. `routes.txt` - Transit routes. A route is a group of trips that are displayed to riders as a single service.
6. `shapes.txt` - Rules for mapping vehicle travel paths, sometimes referred to as route alignments.  
7. `stop_times.txt` - Times that a vehicle arrives at and departs from stops for each trip.  
8. `stops.txt` - Stops where vehicles pick up or drop off riders. Also defines stations and station entrances.
9. `trips.txt` - Trips for each route. A trip is a sequence of two or more stops within a specific time period.    
10. `fare_attributes.txt` - Fare information for a transit agency's routes.
11. `fare_rules.txt` - Rules to apply fares for itineraries.

***10 & 11 are only available in the 2026 GTFS data.**

## Initial Exploration - 2026 GTFS Bus Data
### `stops.txt`

In [1]:
from zipfile import ZipFile
from pathlib import Path
import pandas as pd

# Locate the 2026 bus feed
feed_path = Path("../data/raw/gtfs_bus_2026.zip")

# Open the ZIP file and read stops.txt
with ZipFile(feed_path) as feed:
    stops = pd.read_csv(feed.open("stops.txt"))

# Show the column names
print("Columns:")
print(stops.columns.tolist())

# See if there's any missing items, especailly Lat, Lon!
missing_counts = stops.isna().sum()
print("Missing Counts:")
print(missing_counts)

stops.head(3)

Columns:
['stop_id', 'stop_code', 'stop_name', 'stop_desc', 'stop_lat', 'stop_lon', 'stop_url', 'location_type', 'parent_station', 'tpis_name']
Missing Counts:
stop_id               0
stop_code             0
stop_name             0
stop_desc         11892
stop_lat              0
stop_lon              0
stop_url          11892
location_type     11892
parent_station    11892
tpis_name         11892
dtype: int64


,stop_id,stop_code,stop_name,stop_desc,stop_lat,stop_lon,stop_url,location_type,parent_station,tpis_name
0,1,1,Paramount / Slauson,NaN,33.973248,-118.113113,NaN,NaN,NaN,NaN
1,3,3,Jefferson / 10th,NaN,34.025471,-118.328402,NaN,NaN,NaN,NaN
2,6,6,120th / Augustus F Hawkins,NaN,33.924696,-118.242222,NaN,NaN,NaN,NaN


For `stops.txt`, the main columns that we will be using would be the `stop_id` to merge with other files, as long as the spatial identifier `stop_lat` and `stop_lon` which would be crucial for us to build the geodataframe. `stop_name` and `stop_code` may be essential when it comes to labels in maps for easier references.

### `routes.txt`

In [2]:
# Same steps as above
with ZipFile(feed_path) as feed:
    routes = pd.read_csv(feed.open("routes.txt"))

# Show the column names
print("Columns:")
print(routes.columns.tolist())

# See if there's any missing items
missing_counts = routes.isna().sum()
print("Missing Counts:")
print(missing_counts)

routes.head(3)

Columns:
['route_id', 'route_short_name', 'route_long_name', 'route_desc', 'route_type', 'route_color', 'route_text_color', 'route_url']
Missing Counts:
route_id              0
route_short_name      2
route_long_name       6
route_desc            0
route_type            0
route_color         109
route_text_color    109
route_url           106
dtype: int64


,route_id,route_short_name,route_long_name,route_desc,route_type,route_color,route_text_color,route_url
0,2-13201,2,Metro Local Line,WESTWOOD - EXPOSITION PARK VIA SUNSET-ALVARADO,3,NaN,NaN,NaN
1,4-13201,4,Metro Local Line,DOWNTOWN LA - SANTA MONICA VIA SANTA MONICA BL,3,NaN,NaN,NaN
2,9-13201,Dodger Stadium Express,NaN,This Express Service operates on the dates of ...,3,NaN,NaN,http://www.metro.net/riding/dodger-stadium-exp...


For `routes.txt`, the column that we would utilize the most would be `route_id`, however, other columns provide the neccessary information that we would need to know as well.

### `trips.txt`

In [3]:
# Same steps as above
with ZipFile(feed_path) as feed:
    trips = pd.read_csv(feed.open("trips.txt"))

# Show the column names
print("Columns:")
print(trips.columns.tolist())

# See if there's any missing items
missing_counts = trips.isna().sum()
print("Missing Counts:")
print(missing_counts)

trips.head(5)

Columns:
['route_id', 'service_id', 'trip_id', 'trip_headsign', 'direction_id', 'block_id', 'shape_id', 'trip_id_event']
Missing Counts:
route_id             0
service_id           0
trip_id              0
trip_headsign    33614
direction_id         0
block_id           713
shape_id             0
trip_id_event        0
dtype: int64


,route_id,service_id,trip_id,trip_headsign,direction_id,block_id,shape_id,trip_id_event
0,206-13201,JUNE26-D05CAR-3_Sunday,70206001900850-JUNE26,NaN,1,2065100.0,2060190_JUNE26,63838886-JUNE26-D05CAR-3_Sunday
1,206-13201,JUNE26-D05CAR-3_Sunday,70206001920744-JUNE26,NaN,0,2065400.0,2060192_JUNE26,63838887-JUNE26-D05CAR-3_Sunday
2,206-13201,JUNE26-D03CAR-3_Sunday,70206001901048-JUNE26,NaN,1,2060400.0,2060190_JUNE26,63838888-JUNE26-D03CAR-3_Sunday
3,206-13201,JUNE26-D03CAR-3_Sunday,70206001901323-JUNE26,NaN,1,2060300.0,2060190_JUNE26,63838889-JUNE26-D03CAR-3_Sunday
4,206-13201,JUNE26-D05CAR-3_Sunday,70206001921445-JUNE26,NaN,0,2065300.0,2060192_JUNE26,63838890-JUNE26-D05CAR-3_Sunday


For `trips.txt`, the first three columns would be utilized the most for our initial analysis because these are identifiers that are shared in other data files, which could be used when merging.

### `stop_times.txt`

In [4]:
# Same steps as above
with ZipFile(feed_path) as feed:
    stop_times = pd.read_csv(feed.open("stop_times.txt"))

# Show the column names
print("Columns:")
print(stop_times.columns.tolist())

# See if there's any missing items
missing_counts = stop_times.isna().sum()
print("Missing Counts:")
print(missing_counts)

stop_times.head(5)

/tmp/ipykernel_6348/3617200038.py:3: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  stop_times = pd.read_csv(feed.open("stop_times.txt"))


Columns:
['trip_id', 'arrival_time', 'departure_time', 'stop_id', 'stop_sequence', 'stop_headsign', 'pickup_type', 'drop_off_type', 'trip_id_event', 'route_code', 'destination_code', 'timepoint', 'bay_num']
Missing Counts:
trip_id                   0
arrival_time              0
departure_time            0
stop_id                   0
stop_sequence             0
stop_headsign             0
pickup_type               0
drop_off_type             0
trip_id_event             0
route_code                0
destination_code          0
timepoint                 0
bay_num             2085517
dtype: int64


,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,trip_id_event,route_code,destination_code,timepoint,bay_num
0,70206001900850-JUNE26,08:50:00,08:50:00,2079,1,206 - Vermont / Athens Station,0,0,63838886-JUNE26-D05CAR-3_Sunday,206,Vermont / Athens Station,1,NaN
1,70206001900850-JUNE26,08:51:00,08:51:00,3976,2,206 - Vermont / Athens Station,0,0,63838886-JUNE26-D05CAR-3_Sunday,206,Vermont / Athens Station,0,NaN
2,70206001900850-JUNE26,08:53:00,08:53:00,15434,3,206 - Vermont / Athens Station,0,0,63838886-JUNE26-D05CAR-3_Sunday,206,Vermont / Athens Station,0,NaN
3,70206001900850-JUNE26,08:54:00,08:54:00,11014,4,206 - Vermont / Athens Station,0,0,63838886-JUNE26-D05CAR-3_Sunday,206,Vermont / Athens Station,0,NaN
4,70206001900850-JUNE26,08:55:00,08:55:00,12295,5,206 - Vermont / Athens Station,0,0,63838886-JUNE26-D05CAR-3_Sunday,206,Vermont / Athens Station,0,NaN


For `stop_times.txt`, this file would primarily be used when we are trying to calculate the service frequencies of a stop/area. For our initial analysis of graphing standard accessibility, we would use other files for this part.

### `calendar.txt` & `calendar_dates.txt` 

In [5]:
# Same steps as above
with ZipFile(feed_path) as feed:
    calendar = pd.read_csv(feed.open("calendar.txt"))

# Show the column names
print("Columns:")
print(calendar.columns.tolist())

# See if there's any missing items
missing_counts = calendar.isna().sum()
print("Missing Counts:")
print(missing_counts)

calendar.head(3)

Columns:
['service_id', 'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday', 'start_date', 'end_date']
Missing Counts:
service_id    0
monday        0
tuesday       0
wednesday     0
thursday      0
friday        0
saturday      0
sunday        0
start_date    0
end_date      0
dtype: int64


,service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday,start_date,end_date
0,JUNE26-D01CAR-3_Sunday,0,0,0,0,0,0,1,20260607,20261206
1,JUNE26-D01CAR-1_Weekday,1,1,1,1,1,0,0,20260608,20261211
2,JUNE26-D01CAR-2_Saturday,0,0,0,0,0,1,0,20260613,20261212


In [6]:
with ZipFile(feed_path) as feed:
    calendar_dates = pd.read_csv(feed.open("calendar_dates.txt"))

# Show the column names
print("Columns:")
print(calendar_dates.columns.tolist())

# See if there's any missing items
missing_counts = calendar_dates.isna().sum()
print("Missing Counts:")
print(missing_counts)

calendar_dates.head(3)

Columns:
['service_id', 'date', 'exception_type']
Missing Counts:
service_id        0
date              0
exception_type    0
dtype: int64


,service_id,date,exception_type
0,JUNE26-D01CAR-3_Sunday,20260704,1
1,JUNE26-D01CAR-3_Sunday,20260907,1
2,JUNE26-D01CAR-3_Sunday,20261126,1


The two files provide the necessary service_id information to help us define our study period. Ideally, the study period should be when there are no special service exceptions listed in the `calendar_dates.txt`.

### `shapes.txt`

In [7]:
with ZipFile(feed_path) as feed:
    shapes = pd.read_csv(feed.open("shapes.txt"))

# Show the column names
print("Columns:")
print(shapes.columns.tolist())

# See if there's any missing items
missing_counts = shapes.isna().sum()
print("Missing Counts:")
print(missing_counts)

shapes.head(3)

Columns:
['shape_id', 'shape_pt_lat', 'shape_pt_lon', 'shape_pt_sequence']
Missing Counts:
shape_id             0
shape_pt_lat         0
shape_pt_lon         0
shape_pt_sequence    0
dtype: int64


,shape_id,shape_pt_lat,shape_pt_lon,shape_pt_sequence
0,21150_JUNE26,34.063665,-118.446729,10001
1,21150_JUNE26,34.063663,-118.446610,10002
2,21150_JUNE26,34.063674,-118.445969,10003


For this file, we would only be using it for visualizing the route on the map, if possible. The 7 files we just inspected would be the primary files that we would use for our analysis; the remaining files are for informational purposes.

## Repeat the same process for remainder data

In [21]:
# For convenience, we added bus2026 as well.
paths = {
    "bus2020": "../data/raw/gtfs_bus_2020.zip",
    "bus2026": "../data/raw/gtfs_bus_2026.zip",
    "rail2020": "../data/raw/gtfs_rail_2020.zip",
    "rail2026": "../data/raw/gtfs_rail_2026.zip"
}

files_names = ["stops.txt", "routes.txt", "trips.txt", "stop_times.txt", "calendar.txt", "calendar_dates.txt", "shapes.txt"]

In [18]:
for name, path in paths.items():
    print("="*20)
    print(f"GTFS of: {name}")

    with ZipFile(path) as feed:
        print("="*20)
        for file in files_names:
            data = pd.read_csv(feed.open(file))

            print("="*20)
            print(f"For {file}:")
            print("Rows:", len(data))
            print("Columns:", data.columns.tolist())
            
            print("Missing Counts:")
            print(data.isna().sum().to_dict())

GTFS of: bus2020
For stops.txt:
Rows: 13807
Columns: ['stop_id', 'stop_code', 'stop_name', 'stop_desc', 'stop_lat', 'stop_lon', 'stop_url', 'location_type', 'parent_station', 'tpis_name']
Missing Counts:
{'stop_id': 0, 'stop_code': 0, 'stop_name': 0, 'stop_desc': 13807, 'stop_lat': 0, 'stop_lon': 0, 'stop_url': 13807, 'location_type': 13807, 'parent_station': 13807, 'tpis_name': 13807}
For routes.txt:
Rows: 133
Columns: ['route_id', 'route_short_name', 'route_long_name', 'route_desc', 'route_type', 'route_color', 'route_text_color', 'route_url']
Missing Counts:
{'route_id': 0, 'route_short_name': 2, 'route_long_name': 0, 'route_desc': 0, 'route_type': 0, 'route_color': 112, 'route_text_color': 112, 'route_url': 131}
For trips.txt:
Rows: 27260
Columns: ['route_id', 'service_id', 'trip_id', 'trip_headsign', 'direction_id', 'block_id', 'shape_id']
Missing Counts:
{'route_id': 0, 'service_id': 0, 'trip_id': 0, 'trip_headsign': 27260, 'direction_id': 0, 'block_id': 0, 'shape_id': 0}
For sto

/tmp/ipykernel_6348/1761125149.py:8: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(feed.open(file))


For stop_times.txt:
Rows: 2106179
Columns: ['trip_id', 'arrival_time', 'departure_time', 'stop_id', 'stop_sequence', 'stop_headsign', 'pickup_type', 'drop_off_type', 'trip_id_event', 'route_code', 'destination_code', 'timepoint', 'bay_num']
Missing Counts:
{'trip_id': 0, 'arrival_time': 0, 'departure_time': 0, 'stop_id': 0, 'stop_sequence': 0, 'stop_headsign': 0, 'pickup_type': 0, 'drop_off_type': 0, 'trip_id_event': 0, 'route_code': 0, 'destination_code': 0, 'timepoint': 0, 'bay_num': 2085517}
For calendar.txt:
Rows: 129
Columns: ['service_id', 'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday', 'start_date', 'end_date']
Missing Counts:
{'service_id': 0, 'monday': 0, 'tuesday': 0, 'wednesday': 0, 'thursday': 0, 'friday': 0, 'saturday': 0, 'sunday': 0, 'start_date': 0, 'end_date': 0}
For calendar_dates.txt:
Rows: 7086
Columns: ['service_id', 'date', 'exception_type']
Missing Counts:
{'service_id': 0, 'date': 0, 'exception_type': 0}
For shapes.txt:
Rows: 344948